### Import Dependencies

In [5]:
import openai
from qdrant_client import QdrantClient
from qdrant_client.models import Document, Prefetch, FusionQuery, Filter, FieldCondition, MatchValue
import cohere
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_openai_messages, AIMessage
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display
from typing import Annotated, List, Any
from operator import add
import random
from jinja2 import Template
import instructor
from langsmith import traceable, get_current_run_tree
import numpy as np
import psycopg2
from psycopg2.extras import RealDictCursor

### Add to Shopping Cart Tool

In [2]:
items = [
    {
        "product_id": "B0BH41HYFZ",
        "quantity": 2
    },
    {
        "product_id": "B0BKPB2YQ9",
        "quantity": 4
    }
]

In [6]:
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:

    """Add a list of provided items to the shopping cart.
    
    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.
        
    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
        for item in items:
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(url="http://localhost:6333")

            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id)
                                )
                            ]
                    ),
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            ).points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = 'USD'
        
            # Check if item already exists
            check_query = """
                SELECT id, quantity, price 
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()
            
            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity
                
                update_query = """
                    UPDATE shopping_carts.shopping_cart_items 
                    SET 
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """
                
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                
                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))
            
    return f"Added {items} to the shopping cart."

In [7]:
add_to_shopping_cart(items,'test_user_1','test_cart_1')

"Added [{'product_id': 'B0BH41HYFZ', 'quantity': 2}, {'product_id': 'B0BKPB2YQ9', 'quantity': 4}] to the shopping cart."

In [8]:
items_2 = [
    {
        "product_id": "B0BKPB2YQ9",
        "quantity": 4
    }
]

In [9]:
add_to_shopping_cart(items_2,'test_user_1','test_cart_1')

"Added [{'product_id': 'B0BKPB2YQ9', 'quantity': 4}] to the shopping cart."

### Get Shopping Cart Items Tool

In [10]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:

    """
    Retrieve all items in a user's shopping cart.
    
    Args:
        user_id: User identifier
        cart_id: Cart identifier
    
    Returns:
        List of dictionaries containing cart items
    """
    
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                SELECT 
                    product_id, price, quantity,
                    currency, product_image_url,
                    (price * quantity) as total_price
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s
                ORDER BY added_at DESC
            """
        cursor.execute(query, (user_id, cart_id))

        return [dict(row) for row in cursor.fetchall()]

In [11]:
get_shopping_cart('test_user_1','test_cart_1')

[{'product_id': 'B0BKPB2YQ9',
  'price': Decimal('14.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41WsGRr-3TL._AC_.jpg',
  'total_price': Decimal('119.92')},
 {'product_id': 'B0BH41HYFZ',
  'price': Decimal('26.99'),
  'quantity': 2,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg',
  'total_price': Decimal('53.98')}]

### Delete Shopping Cart Items Tool

In [12]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:

    """
    Remove an item completely from the shopping cart.
    
    Args:
        user_id: User identifier
        product_id: Product identifier to remove
        cart_id: Cart identifier
    
    Returns:
        Information about the removal of the item from the shopping cart.
    """
    
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                DELETE FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
        cursor.execute(query, (user_id, cart_id, product_id))

        return f"Removed {product_id} from the shopping cart." if cursor.rowcount > 0 else f"Item {product_id} not found in the shopping cart."

In [13]:
remove_from_cart('B0BH41HYFZ','test_user_1','test_cart_1')

'Removed B0BH41HYFZ from the shopping cart.'

In [14]:
remove_from_cart('B0BH41HYFZ','test_user_1','test_cart_1')

'Item B0BH41HYFZ not found in the shopping cart.'

In [15]:
get_shopping_cart('test_user_1','test_cart_1')

[{'product_id': 'B0BKPB2YQ9',
  'price': Decimal('14.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41WsGRr-3TL._AC_.jpg',
  'total_price': Decimal('119.92')}]